# Candidate Recommendation — Baseline C2 (Embedding + Cosine)
Query: **Job description → Top-K candidate CVs**

Uses Sentence-Transformers embeddings + cosine similarity.
It also caches resume embeddings to `../models/resume_embeddings_miniLM.npy`.


In [1]:
import os, re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
JOBS_PATH = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/it_jobs.xlsx"
RESUMES_PATH = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv"

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower().replace("\n", " ").replace("\r", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

jobs = pd.read_excel(JOBS_PATH)
resumes = pd.read_csv(RESUMES_PATH)

print("Jobs:", jobs.shape, "Resumes:", resumes.shape)


Jobs: (21961, 35) Resumes: (962, 2)


## Load embedding model

In [3]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## Encode (and cache) resume embeddings (run once)
If you run again later, it will load from cache instead of re-encoding.

In [4]:
os.makedirs("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/models", exist_ok=True)
cache_path = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/models/resume_embeddings_miniLM.npy"

resume_texts = resumes["Resume"].fillna("").apply(clean_text).tolist()

if os.path.exists(cache_path):
    resume_emb = np.load(cache_path)
    print("Loaded cached embeddings:", cache_path, resume_emb.shape)
else:
    resume_emb = model.encode(
        resume_texts,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    np.save(cache_path, resume_emb)
    print("Saved cache:", cache_path, resume_emb.shape)


Batches: 100%|██████████| 16/16 [00:06<00:00,  2.36it/s]

Saved cache: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/models/resume_embeddings_miniLM.npy (962, 384)


## Recommend candidates for one job index

In [5]:
def recommend_candidates_embedding(job_index: int, top_k: int = 10):
    title = str(jobs.loc[job_index, "title"] if "title" in jobs.columns else "")
    desc = str(jobs.loc[job_index, "cleaned_description"] if "cleaned_description" in jobs.columns else "")
    job_text = clean_text(title + " " + desc)

    job_emb = model.encode([job_text], normalize_embeddings=True)
    scores = cosine_similarity(job_emb, resume_emb).flatten()
    top_idx = np.argsort(scores)[::-1][:top_k]

    out = pd.DataFrame({
        "cv_index": top_idx,
        "score": scores[top_idx],
        "Category": resumes.loc[top_idx, "Category"].values if "Category" in resumes.columns else None,
        "resume_preview": resumes.loc[top_idx, "Resume"].apply(lambda s: str(s)[:140] + "...")
    })

    out.insert(0, "job_index", job_index)
    out.insert(1, "job_title", jobs.loc[job_index, "title"] if "title" in jobs.columns else "")
    return out

recommend_candidates_embedding(job_index=0, top_k=10)


,job_index,job_title,cv_index,score,Category,resume_preview
405,0,Support Technologist II 2024-02216,405,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
409,0,Support Technologist II 2024-02216,409,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
413,0,Support Technologist II 2024-02216,413,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
419,0,Support Technologist II 2024-02216,419,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
423,0,Support Technologist II 2024-02216,423,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
427,0,Support Technologist II 2024-02216,427,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
854,0,Support Technologist II 2024-02216,854,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
859,0,Support Technologist II 2024-02216,859,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
864,0,Support Technologist II 2024-02216,864,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
869,0,Support Technologist II 2024-02216,869,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."


## Run a small experiment (multiple jobs) and export CSV

In [6]:
os.makedirs("/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_emb", exist_ok=True)

job_indices = [0, 10, 50]  # change as you like
top_k = 10

all_rows = []
for j in job_indices:
    all_rows.append(recommend_candidates_embedding(j, top_k=top_k))

results = pd.concat(all_rows, ignore_index=True)
out_path = f"/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_emb/candrec_embedding_jobs{len(job_indices)}_top{top_k}.csv"
results.to_csv(out_path, index=False)
print("Saved:", out_path)

results.head(15)


Saved: /Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/outputs/employer/candrec_emb/candrec_embedding_jobs3_top10.csv


,job_index,job_title,cv_index,score,Category,resume_preview
0,0,Support Technologist II 2024-02216,405,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
1,0,Support Technologist II 2024-02216,409,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
2,0,Support Technologist II 2024-02216,413,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
3,0,Support Technologist II 2024-02216,419,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
4,0,Support Technologist II 2024-02216,423,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
5,0,Support Technologist II 2024-02216,427,0.530089,Business Analyst,Key Skills - Requirement Gathering - Requireme...
6,0,Support Technologist II 2024-02216,854,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
7,0,Support Technologist II 2024-02216,859,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
8,0,Support Technologist II 2024-02216,864,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
9,0,Support Technologist II 2024-02216,869,0.503328,Blockchain,"KEY SKILLS: Programing languages: C, C++, Pyth..."
